# Introduction

In these exercises we'll practice renaming columns and combining DataFrames using concatenation and joins.

Run the code cell below to load the data before running the exercises.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 150

country_pool = ['Italy', 'Portugal', 'US', 'Spain', 'France', 'Australia']
variety_pool = ['Pinot Noir', 'Chardonnay', 'Cabernet Sauvignon', 'Riesling',
                'Sangiovese', 'Shiraz']
region_pool = ['Napa Valley', 'Willamette Valley', 'Tuscany', 'Douro',
               'Bordeaux', 'Barossa Valley', 'Marlborough', 'Rioja']

countries = list(np.random.choice(country_pool, n))
varieties = list(np.random.choice(variety_pool, n))
points = list(np.random.randint(82, 98, n))
prices = [round(float(p), 1) for p in np.random.uniform(10, 200, n)]
regions_1 = list(np.random.choice(region_pool, n))
regions_2 = [f'Sub-region {i % 6}' for i in range(n)]

taster_names = ['Roger Voss', 'Paul Gregutt', "Kerin O'Keefe",
                'Anna Lee C. Iijima', 'Michael Schachner', 'Joe Czerwinski']
taster_handles = ['@vossroger', '@paulgwine', '@kerinokeefe',
                  '@annaiijima', '@wineschach', '@JoeCz']
taster_name_col = [taster_names[i % 6] for i in range(n)]
taster_handle_col = [taster_handles[i % 6] for i in range(n)]

reviews = pd.DataFrame({
    'country': countries,
    'description': [f'Wine description for review {i}.' for i in range(n)],
    'designation': [None if i % 4 == 0 else f'Label {i % 15}' for i in range(n)],
    'points': points,
    'price': prices,
    'province': [f'Province {i % 8}' for i in range(n)],
    'region_1': regions_1,
    'region_2': regions_2,
    'taster_name': taster_name_col,
    'taster_twitter_handle': taster_handle_col,
    'title': [f'Winery {i % 20} {2010 + i % 10} Vintage Wine' for i in range(n)],
    'variety': varieties,
    'winery': [f'Winery {i % 20}' for i in range(n)],
})

# Two subsets used in the combining exercises
italian_wines = reviews[reviews.country == 'Italy'].copy().reset_index(drop=True)
us_wines = reviews[reviews.country == 'US'].copy().reset_index(drop=True)

# Lookup table used in the join exercise
critics = pd.DataFrame({
    'years_experience': [20, 12, 15, 8, 10, 18],
    'specialty': ['Old World Reds', 'Pacific NW', 'Italian Whites',
                  'German Riesling', 'South American', 'Bordeaux Blends'],
}, index=taster_names)

pd.set_option('display.max_rows', 5)
print(f'Main dataset: {reviews.shape}')
print(f'Italian wines subset: {italian_wines.shape}')
print(f'US wines subset: {us_wines.shape}')
print('Setup complete.')
reviews.head()

Main dataset: (150, 13)
Italian wines subset: (24, 13)
US wines subset: (23, 13)
Setup complete.


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Spain,Wine description for review 0.,NaN,97,178.5,Province 0,Napa Valley,Sub-region 0,Roger Voss,@vossroger,Winery 0 2010 Vintage Wine,Sangiovese,Winery 0
1,France,Wine description for review 1.,Label 1,95,15.2,Province 1,Willamette Valley,Sub-region 1,Paul Gregutt,@paulgwine,Winery 1 2011 Vintage Wine,Sangiovese,Winery 1
2,US,Wine description for review 2.,Label 2,90,120.0,Province 2,Willamette Valley,Sub-region 2,Kerin O'Keefe,@kerinokeefe,Winery 2 2012 Vintage Wine,Cabernet Sauvignon,Winery 2
3,France,Wine description for review 3.,Label 3,84,93.3,Province 3,Douro,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 3 2013 Vintage Wine,Sangiovese,Winery 3
4,France,Wine description for review 4.,NaN,90,137.7,Province 4,Napa Valley,Sub-region 4,Michael Schachner,@wineschach,Winery 4 2014 Vintage Wine,Riesling,Winery 4


# Exercises

## 1.

`region_1` and `region_2` are pretty uninformative column names. Rename these columns to `region` and `locale` respectively, and assign the result to `renamed`.

In [2]:
renamed = reviews.rename(columns={'region_1': 'region', 'region_2': 'locale'})
print(renamed.columns.tolist())
renamed.head()

['country', 'description', 'designation', 'points', 'price', 'province', 'region', 'locale', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']


,country,description,designation,points,price,province,region,locale,taster_name,taster_twitter_handle,title,variety,winery
0,Spain,Wine description for review 0.,NaN,97,178.5,Province 0,Napa Valley,Sub-region 0,Roger Voss,@vossroger,Winery 0 2010 Vintage Wine,Sangiovese,Winery 0
1,France,Wine description for review 1.,Label 1,95,15.2,Province 1,Willamette Valley,Sub-region 1,Paul Gregutt,@paulgwine,Winery 1 2011 Vintage Wine,Sangiovese,Winery 1
2,US,Wine description for review 2.,Label 2,90,120.0,Province 2,Willamette Valley,Sub-region 2,Kerin O'Keefe,@kerinokeefe,Winery 2 2012 Vintage Wine,Cabernet Sauvignon,Winery 2
3,France,Wine description for review 3.,Label 3,84,93.3,Province 3,Douro,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 3 2013 Vintage Wine,Sangiovese,Winery 3
4,France,Wine description for review 4.,NaN,90,137.7,Province 4,Napa Valley,Sub-region 4,Michael Schachner,@wineschach,Winery 4 2014 Vintage Wine,Riesling,Winery 4


**Key insight:** `.rename()` accepts a `columns=` dict mapping `{old_name: new_name}`. You only need to list the columns you're changing — all others are left untouched. It returns a **new** DataFrame; the original `reviews` is unchanged.

To rename index labels instead: `.rename(index={old_label: new_label})`.

## 2.

Set the index name in the `reviews` dataset to `wines`. Assign the result to `reindexed`.

In [3]:
reindexed = reviews.rename_axis('wines', axis='rows')
print(f'Index name: {reindexed.index.name}')
reindexed.head()

Index name: wines


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
wines,,,,,,,,,,,,,
0,Spain,Wine description for review 0.,NaN,97,178.5,Province 0,Napa Valley,Sub-region 0,Roger Voss,@vossroger,Winery 0 2010 Vintage Wine,Sangiovese,Winery 0
1,France,Wine description for review 1.,Label 1,95,15.2,Province 1,Willamette Valley,Sub-region 1,Paul Gregutt,@paulgwine,Winery 1 2011 Vintage Wine,Sangiovese,Winery 1
2,US,Wine description for review 2.,Label 2,90,120.0,Province 2,Willamette Valley,Sub-region 2,Kerin O'Keefe,@kerinokeefe,Winery 2 2012 Vintage Wine,Cabernet Sauvignon,Winery 2
3,France,Wine description for review 3.,Label 3,84,93.3,Province 3,Douro,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 3 2013 Vintage Wine,Sangiovese,Winery 3
4,France,Wine description for review 4.,NaN,90,137.7,Province 4,Napa Valley,Sub-region 4,Michael Schachner,@wineschach,Winery 4 2014 Vintage Wine,Riesling,Winery 4


**Key insight:** `.rename_axis()` sets the **name of the axis** (the label that appears above the index when printed), not the index values themselves.

- `axis='rows'` (or `axis=0`) → names the row index
- `axis='columns'` (or `axis=1`) → names the column axis

This is distinct from `.rename(index=...)` which renames specific index *values*.

## 3.

The Itlian and US wine subsets are already created for you as `italian_wines` and `us_wines`. Create a `combined` DataFrame by concatenating them, and reset the index so it runs from 0 to len(combined)-1.

In [4]:
combined = pd.concat([italian_wines, us_wines])
print(f'Italian wines: {len(italian_wines)} rows')
print(f'US wines:      {len(us_wines)} rows')
print(f'Combined:      {len(combined)} rows')
combined.head()

Italian wines: 24 rows
US wines:      23 rows
Combined:      47 rows


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,Wine description for review 21.,Label 6,86,88.2,Province 5,Tuscany,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 1 2011 Vintage Wine,Riesling,Winery 1
1,Italy,Wine description for review 27.,Label 12,88,17.8,Province 3,Douro,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 7 2017 Vintage Wine,Pinot Noir,Winery 7
2,Italy,Wine description for review 28.,NaN,84,85.8,Province 4,Napa Valley,Sub-region 4,Michael Schachner,@wineschach,Winery 8 2018 Vintage Wine,Shiraz,Winery 8
3,Italy,Wine description for review 40.,NaN,95,27.8,Province 0,Douro,Sub-region 4,Michael Schachner,@wineschach,Winery 0 2010 Vintage Wine,Cabernet Sauvignon,Winery 0
4,Italy,Wine description for review 45.,Label 0,87,134.9,Province 5,Willamette Valley,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 5 2015 Vintage Wine,Cabernet Sauvignon,Winery 5


**Key insight:** `pd.concat([df1, df2])` stacks DataFrames **vertically** (by rows). The original index values are preserved by default, which can produce duplicate index values. Common patterns:

```python
pd.concat([df1, df2])                          # preserve original indices
pd.concat([df1, df2], ignore_index=True)       # reset to 0,1,2,...
pd.concat([df1, df2], axis=1)                  # stack horizontally (by columns)
pd.concat([df1, df2], keys=['italy', 'us'])    # add outer MultiIndex level
```

## 4.

A `critics` lookup table has been created for you, indexed by taster name. Join it to the `reviews` DataFrame so each review also shows the critic's `years_experience` and `specialty`. Assign the result to `enriched`.

In [5]:
enriched = reviews.join(critics, on='taster_name')
print(f'Columns added: {[c for c in enriched.columns if c not in reviews.columns]}')
enriched[['taster_name', 'points', 'years_experience', 'specialty']].head(8)

Columns added: ['years_experience', 'specialty']


,taster_name,points,years_experience,specialty
0,Roger Voss,97,20,Old World Reds
1,Paul Gregutt,95,12,Pacific NW
...,...,...,...,...
6,Roger Voss,83,20,Old World Reds
7,Paul Gregutt,97,12,Pacific NW


**Key insight:** `.join()` merges on the **index** of the right DataFrame by default. Use `on=` to specify which column in the *left* DataFrame to match against the right index.

| Method | Joins on | Best for |
|---|---|---|
| `df.join(other, on='col')` | left col ↔ right index | lookup tables indexed by a key |
| `pd.merge(df1, df2, on='col')` | shared column in both | two DataFrames with a common key column |
| `pd.concat([df1, df2])` | no key — stacks rows | combining datasets with identical schema |

Default join type is **left** — all rows in `reviews` are kept; unmatched critics get NaN.

---
## Session Summary — Day 6 Pandas

| Exercise | Topic | Key Concept |
|----------|-------|-------------|
| 1 | Rename columns | `.rename(columns={old: new})` — only list columns to change; returns new DataFrame |
| 2 | Rename index axis | `.rename_axis('name', axis='rows')` — names the axis label, not the values |
| 3 | Concatenate DataFrames | `pd.concat([df1, df2])` — stacks rows; use `ignore_index=True` to reset index |
| 4 | Join on a lookup table | `df.join(lookup, on='col')` — match left column against right index; left join by default |